# GTEx Z-Score Analysis

In this notebook, we calculate a Z-score for each gene across the GTEx tissues. The Z-score shows whether the expression of a gene in one tissue is higher or lower than the average expression of that gene across all tissues.

We use the method described in the research paper **“Enrichr: interactive and collaborative HTML5 gene list enrichment analysis tool” (2013)**, where the average and standard deviation are calculated for each gene and a Z-score is calculated for each gene-expression value. Genes with an absolute Z-score greater than 3 are considered to have a strong expression signal.

We will classify the results as:

- `Z > 3` = Up
- `Z < -3` = Down
- `-3 <= Z <= 3` = Not strongly different

In [2]:
import pandas as pd
import numpy as np

In [3]:
# load the GTEx median TPM dataset
file_name = "../data/GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_median_tpm.gct.gz"

df = pd.read_csv(
    file_name,
    sep="\t",
    skiprows=2
)

In [4]:
# get all the tissue columns
tissue_cols = df.columns[2:]

In [5]:
# create the gene-tissue table
gene_tissue_table = df[
    ["Name", "Description"] + list(tissue_cols)
].copy()

In [6]:
# get only the tissue expression values
expression_table = gene_tissue_table[tissue_cols]

# calculate the average expression for each gene
gene_mean = expression_table.mean(axis=1)

# calculate the standard deviation for each gene
gene_std = expression_table.std(axis=1)

In [7]:
# calculate the z-score for each gene in each tissue
z_scores = expression_table.sub(
    gene_mean,
    axis=0
).div(
    gene_std,
    axis=0
)

z_scores.head()

,Adipose_Subcutaneous,Adipose_Visceral_Omentum,Adrenal_Gland,Artery_Aorta,Artery_Coronary,Artery_Tibial,Bladder,Brain_Amygdala,Brain_Anterior_cingulate_cortex_BA24,Brain_Caudate_basal_ganglia,...,Spleen,Stomach,Stomach_Mixed_Cell,Stomach_Mucosa,Stomach_Muscularis,Testis,Thyroid,Uterus,Vagina,Whole_Blood
0,-0.432200,-0.432200,-0.432200,-0.432200,-0.432200,-0.432200,-0.432200,0.205165,-0.432200,0.141807,...,-0.432200,-0.432200,0.375490,0.252345,0.433004,3.528795,-0.432200,-0.432200,-0.432200,-0.432200
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-0.041377,-0.402930,-0.503538,-0.131276,-0.196052,-0.560924,0.415621,-1.285441,-1.227729,-1.207887,...,1.664754,-0.310818,-0.427218,-0.664910,-0.400263,0.401435,0.985264,1.586742,0.882957,-0.202400
3,-0.246708,-0.246708,-0.246708,-0.246708,-0.246708,-0.246708,-0.246708,-0.246708,-0.246708,-0.246708,...,-0.246708,-0.246708,-0.246708,-0.246708,-0.246708,4.231393,-0.246708,-0.246708,-0.246708,-0.246708
4,-0.480652,-0.480652,-0.480652,-0.480652,-0.480652,-0.480652,-0.480652,0.328580,0.364574,0.304259,...,-0.480652,-0.480652,0.483693,0.343116,0.662592,-0.480652,-0.480652,-0.480652,-0.480652,-0.480652


In [9]:
# create a table for the classifications
classification = pd.DataFrame(
    "Not strongly different",
    index=z_scores.index,
    columns=z_scores.columns
)

# classify each z-score
classification[z_scores > 3] = "Up"
classification[z_scores < -3] = "Down"

classification.head()

,Adipose_Subcutaneous,Adipose_Visceral_Omentum,Adrenal_Gland,Artery_Aorta,Artery_Coronary,Artery_Tibial,Bladder,Brain_Amygdala,Brain_Anterior_cingulate_cortex_BA24,Brain_Caudate_basal_ganglia,...,Spleen,Stomach,Stomach_Mixed_Cell,Stomach_Mucosa,Stomach_Muscularis,Testis,Thyroid,Uterus,Vagina,Whole_Blood
0,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,...,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Up,Not strongly different,Not strongly different,Not strongly different,Not strongly different
1,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,...,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different
2,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,...,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different
3,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,...,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Up,Not strongly different,Not strongly different,Not strongly different,Not strongly different
4,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,...,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different,Not strongly different


In [10]:
# turn the z-score table into a long format
z_long = z_scores.reset_index().melt(
    id_vars="index",
    var_name="Tissue",
    value_name="Z_score"
)

# turn the classification table into a long format
class_long = classification.reset_index().melt(
    id_vars="index",
    var_name="Tissue",
    value_name="Classification"
)

# combine both tables
z_results = z_long.copy()
z_results["Classification"] = class_long["Classification"]

# keep only strong signals
strong_results = z_results[
    z_results["Classification"] != "Not strongly different"
]

strong_results.head()

,index,Tissue,Z_score,Classification
49,49,Adipose_Subcutaneous,3.233330,Up
433,433,Adipose_Subcutaneous,4.186634,Up
787,787,Adipose_Subcutaneous,4.509170,Up
1692,1692,Adipose_Subcutaneous,3.069906,Up
1841,1841,Adipose_Subcutaneous,3.273816,Up


In [11]:
# count the classifications
strong_results["Classification"].value_counts()

Classification
Up      84847
Down        4
Name: count, dtype: int64

In this analysis, we calculated the average expression and standard deviation for each gene across all GTEx tissues. We then used these values to calculate a Z-score for every gene-tissue pair. After that, we classified the results as Up when the Z-score was greater than 3, Down when it was less than -3, and Not strongly different when it was between -3 and 3. Finally, we filtered the results to focus on the genes with strong Up or Down expression signals.